<a href="https://colab.research.google.com/github/ntlcs/fiap-tech-challenge-fase-3/blob/main/03_Desafio_FIAP_IA_05_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tech Challenge - Fase 3

## RAG - Recuperação de protocolos clínicos

Nesta etapa será implementado um pipeline de Retrieval-Augmented Generation (RAG).

O objetivo é permitir que o assistente consulte protocolos institucionais antes de gerar uma resposta clínica.

A recuperação dos documentos será realizada por similaridade semântica.

Cada documento possuirá metadados de origem, permitindo que as respostas futuras apresentem as fontes utilizadas.

Os protocolos utilizados nesta prova de conceito são sintéticos e possuem finalidade exclusivamente acadêmica.

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

PROTOCOL_PATH = (
    PROJECT_DIR
    / "data"
    / "synthetic"
    / "protocolos_sinteticos.csv"
)

VECTOR_DB_DIR = (
    PROJECT_DIR
    / "data"
    / "database"
    / "faiss_protocolos"
)

print("Protocolos:", PROTOCOL_PATH)
print("Arquivo existe:", PROTOCOL_PATH.exists())

Protocolos: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/synthetic/protocolos_sinteticos.csv
Arquivo existe: True


In [ ]:
df_protocolos = pd.read_csv(PROTOCOL_PATH)

print("Quantidade de protocolos:", len(df_protocolos))
print()
print(df_protocolos.columns.tolist())

df_protocolos

Quantidade de protocolos: 4

['protocolo_id', 'titulo', 'conteudo']


,protocolo_id,titulo,conteudo
0,PROTO-DM-001,Monitoramento do controle glicêmico,Pacientes com Diabetes Mellitus Tipo 2 devem t...
1,PROTO-DM-002,Avaliação renal,Pacientes com Diabetes Mellitus Tipo 2 devem s...
2,PROTO-DM-003,Avaliação oftalmológica,O acompanhamento de pessoas com Diabetes Melli...
3,PROTO-DM-004,Segurança do assistente clínico,O assistente virtual deve atuar exclusivamente...


In [ ]:
for indice, linha in df_protocolos.iterrows():
    print(f"PROTOCOLO {indice + 1}")
    print(linha.to_dict())
    print("-" * 80)

PROTOCOLO 1
{'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico', 'conteudo': 'Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.'}
--------------------------------------------------------------------------------
PROTOCOLO 2
{'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal', 'conteudo': 'Pacientes com Diabetes Mellitus Tipo 2 devem ser acompanhados quanto à função renal. Exames como creatinina e avaliação de albuminúria podem fazer parte do acompanhamento conforme avaliação médica.'}
--------------------------------------------------------------------------------
PROTOCOLO 3
{'protocolo_id': 'PROTO-DM-003', 'titulo': 'Avaliação oftalmológica', 'conteudo': 'O acompanhamento de pessoas com Diabetes Mellitus Tipo 2 pode incluir avaliação oftalmológ

In [ ]:
from langchain_core.documents import Document

documentos = []

for indice, linha in df_protocolos.iterrows():

    documento = Document(
        page_content=linha["conteudo"],
        metadata={
            "protocolo_id": linha["protocolo_id"],
            "titulo": linha["titulo"],
            "fonte": "Protocolo institucional sintético",
            "tipo": "protocolo_clinico"
        }
    )

    documentos.append(documento)

print("Documentos criados:", len(documentos))

Documentos criados: 4


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Modelo de embeddings carregado.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings carregado.


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documentos,
    embedding_model
)

print("Índice FAISS criado com sucesso.")

/tmp/ipykernel_3015/1606724547.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Índice FAISS criado com sucesso.


In [ ]:
VECTOR_DB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

vectorstore.save_local(
    str(VECTOR_DB_DIR)
)

print("Índice salvo em:", VECTOR_DB_DIR)

Índice salvo em: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/data/database/faiss_protocolos


In [ ]:
consulta = """
Quais aspectos devem ser acompanhados em um paciente
com Diabetes Mellitus Tipo 2?
"""

resultados = vectorstore.similarity_search(
    consulta,
    k=2
)

print("Resultados encontrados:", len(resultados))

Resultados encontrados: 2


In [ ]:
for i, doc in enumerate(resultados, start=1):

    print(f"RESULTADO {i}")
    print("Metadata:", doc.metadata)
    print()
    print(doc.page_content)
    print("-" * 80)

RESULTADO 1
Metadata: {'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}

Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.
--------------------------------------------------------------------------------
RESULTADO 2
Metadata: {'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}

Pacientes com Diabetes Mellitus Tipo 2 devem ser acompanhados quanto à função renal. Exames como creatinina e avaliação de albuminúria podem fazer parte do acompanhamento conforme avaliação médica.
--------------------------------------------------------------------------------


In [ ]:
consulta = """
Paciente com Diabetes Mellitus Tipo 2 possui avaliação
oftalmológica pendente. O que deve ser acompanhado?
"""

resultados = vectorstore.similarity_search(
    consulta,
    k=2
)

for i, doc in enumerate(resultados, start=1):

    print(f"RESULTADO {i}")
    print(doc.metadata)
    print()
    print(doc.page_content)
    print("-" * 80)

RESULTADO 1
{'protocolo_id': 'PROTO-DM-003', 'titulo': 'Avaliação oftalmológica', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}

O acompanhamento de pessoas com Diabetes Mellitus Tipo 2 pode incluir avaliação oftalmológica periódica para rastreamento de alterações relacionadas à doença.
--------------------------------------------------------------------------------
RESULTADO 2
{'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}

Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.
--------------------------------------------------------------------------------


In [ ]:
def buscar_protocolos(pergunta, k=2):

    resultados = vectorstore.similarity_search(
        pergunta,
        k=k
    )

    return resultados

In [ ]:
documentos_encontrados = buscar_protocolos(
    "Quais cuidados devem ser considerados no acompanhamento do DM2?"
)

for doc in documentos_encontrados:
    print(doc.metadata)
    print(doc.page_content)
    print()

{'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}
Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.

{'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal', 'fonte': 'Protocolo institucional sintético', 'tipo': 'protocolo_clinico'}
Pacientes com Diabetes Mellitus Tipo 2 devem ser acompanhados quanto à função renal. Exames como creatinina e avaliação de albuminúria podem fazer parte do acompanhamento conforme avaliação médica.



In [ ]:
def formatar_contexto_protocolos(documentos):

    blocos = []

    for indice, doc in enumerate(documentos, start=1):

        fonte = doc.metadata.get(
            "titulo",
            "Protocolo institucional"
        )

        bloco = (
            f"[Fonte {indice}: {fonte}]\n"
            f"{doc.page_content}"
        )

        blocos.append(bloco)

    return "\n\n".join(blocos)

In [ ]:
documentos_encontrados = buscar_protocolos(
    "acompanhamento de paciente com diabetes tipo 2"
)

contexto_protocolos = formatar_contexto_protocolos(
    documentos_encontrados
)

print(contexto_protocolos)

[Fonte 1: Monitoramento do controle glicêmico]
Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.

[Fonte 2: Avaliação oftalmológica]
O acompanhamento de pessoas com Diabetes Mellitus Tipo 2 pode incluir avaliação oftalmológica periódica para rastreamento de alterações relacionadas à doença.


In [ ]:
print(df_protocolos.columns.tolist())

['protocolo_id', 'titulo', 'conteudo']


In [ ]:
for indice, linha in df_protocolos.iterrows():
    print(linha.to_dict())

{'protocolo_id': 'PROTO-DM-001', 'titulo': 'Monitoramento do controle glicêmico', 'conteudo': 'Pacientes com Diabetes Mellitus Tipo 2 devem ter o controle glicêmico acompanhado periodicamente por meio de avaliação clínica e exames laboratoriais. Alterações persistentes devem ser analisadas pelo médico responsável considerando o histórico individual.'}
{'protocolo_id': 'PROTO-DM-002', 'titulo': 'Avaliação renal', 'conteudo': 'Pacientes com Diabetes Mellitus Tipo 2 devem ser acompanhados quanto à função renal. Exames como creatinina e avaliação de albuminúria podem fazer parte do acompanhamento conforme avaliação médica.'}
{'protocolo_id': 'PROTO-DM-003', 'titulo': 'Avaliação oftalmológica', 'conteudo': 'O acompanhamento de pessoas com Diabetes Mellitus Tipo 2 pode incluir avaliação oftalmológica periódica para rastreamento de alterações relacionadas à doença.'}
{'protocolo_id': 'PROTO-DM-004', 'titulo': 'Segurança do assistente clínico', 'conteudo': 'O assistente virtual deve atuar excl